In [1]:
import logging
import sys
from datetime import datetime

log_filename = "logs.txt"
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(message)s',
    handlers=[
        logging.FileHandler(log_filename),
        logging.StreamHandler(sys.stdout)
    ]
)

logger = logging.getLogger(__name__)
logger.info("="*80)
logger.info(f"Iris Classification - Optimization in Neural Networks")
logger.info(f"Start time: {datetime.now()}")
logger.info("="*80)

output_dir = "visualizations/"
import os
os.makedirs(output_dir, exist_ok=True)

2026-04-02 17:56:17,010 - ================================================================================
2026-04-02 17:56:17,011 - Iris Classification - Optimization in Neural Networks
2026-04-02 17:56:17,011 - Start time: 2026-04-02 17:56:17.011734
2026-04-02 17:56:17,012 - ================================================================================


# Iris Classification - Optimization in Neural Networks

# Requirements

1) Install Library

In [2]:
!pip install ucimlrepo -q

import copy
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from collections import OrderedDict
from torch.func import functional_call, jacrev
from torch.nn.utils import parameters_to_vector, vector_to_parameters
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from ucimlrepo import fetch_ucirepo

2026-04-02 17:56:21,710 - NumExpr defaulting to 4 threads.


2) Set Seed

In [3]:
seedValue = 42
np.random.seed(seedValue)
torch.manual_seed(seedValue)

logger.info(f"Torch version: {torch.__version__}")
logger.info(f"Seed: {seedValue}")

2026-04-02 17:56:27,488 - Torch version: 2.10.0+cu128
2026-04-02 17:56:27,489 - Seed: 42


# Dataset

3) Load the dataset

In [4]:
irisData = fetch_ucirepo(id=53)

xDataFrame = irisData.data.features.copy()
yDataFrame = irisData.data.targets.copy()

print("Metadata:")
print(irisData.metadata)

print("\nVariable Information:")
print(irisData.variables)

print("\nFeature Sample:")
display(xDataFrame.head())

print("\nTarget Sample:")
display(yDataFrame.head())

Metadata:
{'uci_id': 53, 'name': 'Iris', 'repository_url': 'https://archive.ics.uci.edu/dataset/53/iris', 'data_url': 'https://archive.ics.uci.edu/static/public/53/data.csv', 'abstract': 'A small classic dataset from Fisher, 1936. One of the earliest known datasets used for evaluating classification methods.\n', 'area': 'Biology', 'tasks': ['Classification'], 'characteristics': ['Tabular'], 'num_instances': 150, 'num_features': 4, 'feature_types': ['Real'], 'demographics': [], 'target_col': ['class'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 1936, 'last_updated': 'Tue Sep 12 2023', 'dataset_doi': '10.24432/C56C76', 'creators': ['R. A. Fisher'], 'intro_paper': {'ID': 191, 'type': 'NATIVE', 'title': 'The Iris data set: In search of the source of virginica', 'authors': 'A. Unwin, K. Kleinman', 'venue': 'Significance, 2021', 'year': 2021, 'journal': 'Significance, 2021', 'DOI': '1740-9713.01589', 'URL': 'https://www.semantics

,sepal length,sepal width,petal length,petal width
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2



Target Sample:


,class
0,Iris-setosa
1,Iris-setosa
2,Iris-setosa
3,Iris-setosa
4,Iris-setosa


4) Normalize input features

In [5]:
featureScaler = StandardScaler()
xNormalizedArray = featureScaler.fit_transform(xDataFrame.values).astype(np.float32)

logger.info(f"Feature shape after normalization: {xNormalizedArray.shape}")
logger.info("First 5 normalized feature rows:")
logger.info(f"\n{xNormalizedArray[:5]}")

2026-04-02 17:56:28,245 - Feature shape after normalization: (150, 4)
2026-04-02 17:56:28,246 - First 5 normalized feature rows:
2026-04-02 17:56:28,247 - 
[[-0.9006812   1.0320572  -1.3412724  -1.3129767 ]
 [-1.1430169  -0.1249576  -1.3412724  -1.3129767 ]
 [-1.3853526   0.33784834 -1.3981382  -1.3129767 ]
 [-1.5065205   0.10644536 -1.2844067  -1.3129767 ]
 [-1.021849    1.2634602  -1.3412724  -1.3129767 ]]


5) Apply one-hot encoding to labels

In [6]:
labelEncoder = LabelEncoder()
yLabelArray = labelEncoder.fit_transform(yDataFrame.iloc[:, 0])

classNames = list(labelEncoder.classes_)
classCount = len(classNames)

yOneHotArray = np.eye(classCount, dtype=np.float32)[yLabelArray]

logger.info(f"Class Names: {classNames}")
logger.info(f"Integer labels (first 5): {yLabelArray[:5]}")
logger.info(f"One-hot labels (first 5):\n{yOneHotArray[:5]}")

2026-04-02 17:56:28,287 - Class Names: ['Iris-setosa', 'Iris-versicolor', 'Iris-virginica']
2026-04-02 17:56:28,288 - Integer labels (first 5): [0 0 0 0 0]
2026-04-02 17:56:28,289 - One-hot labels (first 5):
[[1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]]


6) Split into training and testing sets

In [7]:
xTrainArray, xTestArray, yTrainArray, yTestArray = train_test_split(
    xNormalizedArray,
    yOneHotArray,
    test_size=0.2,
    random_state=seedValue,
    stratify=yLabelArray
)

xTrainTensor = torch.tensor(xTrainArray, dtype=torch.float32)
xTestTensor = torch.tensor(xTestArray, dtype=torch.float32)
yTrainTensor = torch.tensor(yTrainArray, dtype=torch.float32)
yTestTensor = torch.tensor(yTestArray, dtype=torch.float32)

print("xTrainTensor shape:", xTrainTensor.shape)
print("xTestTensor shape:", xTestTensor.shape)
print("yTrainTensor shape:", yTrainTensor.shape)
print("yTestTensor shape:", yTestTensor.shape)

xTrainTensor shape: torch.Size([120, 4])
xTestTensor shape: torch.Size([30, 4])
yTrainTensor shape: torch.Size([120, 3])
yTestTensor shape: torch.Size([30, 3])


# Model Architecture

7) Construct a feedforward neural network with input 4, hidden 8, hidden 6, output 3

In [8]:
def buildSequentialModel():
    modelInstance = nn.Sequential(
        OrderedDict([
            ("inputLayer", nn.Linear(4, 8)),
            ("reluLayer1", nn.ReLU()),
            ("hiddenLayer1", nn.Linear(8, 6)),
            ("reluLayer2", nn.ReLU()),
            ("outputLayer", nn.Linear(6, 3)),
            ("softmaxLayer", nn.Softmax(dim=1))
        ])
    )
    return modelInstance

baseModel = buildSequentialModel()
print(baseModel)

Sequential(
  (inputLayer): Linear(in_features=4, out_features=8, bias=True)
  (reluLayer1): ReLU()
  (hiddenLayer1): Linear(in_features=8, out_features=6, bias=True)
  (reluLayer2): ReLU()
  (outputLayer): Linear(in_features=6, out_features=3, bias=True)
  (softmaxLayer): Softmax(dim=1)
)


8) Implement the model using a sequential approach

In [9]:
gradientDescentModel = buildSequentialModel()
gaussNewtonModel = buildSequentialModel()

_ = gradientDescentModel(xTrainTensor[:1])
_ = gaussNewtonModel(xTrainTensor[:1])

initialStateDict = copy.deepcopy(gradientDescentModel.state_dict())
gaussNewtonModel.load_state_dict(initialStateDict)

print("Gradient Descent Model:")
print(gradientDescentModel)

print("\nGauss-Newton Model:")
print(gaussNewtonModel)

Gradient Descent Model:
Sequential(
  (inputLayer): Linear(in_features=4, out_features=8, bias=True)
  (reluLayer1): ReLU()
  (hiddenLayer1): Linear(in_features=8, out_features=6, bias=True)
  (reluLayer2): ReLU()
  (outputLayer): Linear(in_features=6, out_features=3, bias=True)
  (softmaxLayer): Softmax(dim=1)
)

Gauss-Newton Model:
Sequential(
  (inputLayer): Linear(in_features=4, out_features=8, bias=True)
  (reluLayer1): ReLU()
  (hiddenLayer1): Linear(in_features=8, out_features=6, bias=True)
  (reluLayer2): ReLU()
  (outputLayer): Linear(in_features=6, out_features=3, bias=True)
  (softmaxLayer): Softmax(dim=1)
)


# Forward Model

9) Compute the predicted output for the training data

In [10]:
with torch.no_grad():
    yTrainPredInitialTensor = gradientDescentModel(xTrainTensor)

print("Predicted output training data (first 5):")
print(yTrainPredInitialTensor[:5])

Predicted output training data (first 5):
tensor([[0.4265, 0.3098, 0.2637],
        [0.4179, 0.3145, 0.2676],
        [0.4505, 0.2940, 0.2555],
        [0.4371, 0.3013, 0.2616],
        [0.4326, 0.3049, 0.2625]])


# Loss Function

10) Implement the loss function

In [11]:
def categoricalCrossEntropy(yTrueTensor, yPredTensor):
    yPredTensor = torch.clamp(yPredTensor, min=1e-7, max=1.0)
    lossValue = -(yTrueTensor * torch.log(yPredTensor)).sum(dim=1).mean()
    return lossValue

initialLossValue = categoricalCrossEntropy(yTrainTensor, yTrainPredInitialTensor)
print("Initial training loss:", initialLossValue.item())

Initial training loss: 1.1184314489364624


# Full Batch Gradient Descent

11) Use the entire dataset (full batch)

In [12]:
xFullBatchTensor = xTrainTensor
yFullBatchTensor = yTrainTensor

print("Number of full batch samples:", xFullBatchTensor.shape[0])
print("Number of features:", xFullBatchTensor.shape[1])

Number of full batch samples: 120
Number of features: 4


12) Compute gradients using automatic differentiation

In [13]:
gradientDescentModel.zero_grad()

yPredTensor = gradientDescentModel(xFullBatchTensor)
lossValue = categoricalCrossEntropy(yFullBatchTensor, yPredTensor)
lossValue.backward()

for parameterName, parameterValue in gradientDescentModel.named_parameters():
    print(parameterName, parameterValue.grad.shape)

inputLayer.weight torch.Size([8, 4])
inputLayer.bias torch.Size([8])
hiddenLayer1.weight torch.Size([6, 8])
hiddenLayer1.bias torch.Size([6])
outputLayer.weight torch.Size([3, 6])
outputLayer.bias torch.Size([3])


13) Update all parameters manually

In [14]:
learningRateValue = 0.5

with torch.no_grad():
    for parameterValue in gradientDescentModel.parameters():
        parameterValue -= learningRateValue * parameterValue.grad

updatedPredTensor = gradientDescentModel(xTrainTensor)
updatedLossValue = categoricalCrossEntropy(yTrainTensor, updatedPredTensor)

print("Loss before update:", lossValue.item())
print("Loss after update:", updatedLossValue.item())

Loss before update: 1.1184314489364624
Loss after update: 1.1011407375335693


# Helper Functions for Training, Accuracy, and Parameter Utilities

14) Helper functions

In [15]:
def calculateAccuracy(yTrueTensor, yPredTensor):
    trueClassTensor = torch.argmax(yTrueTensor, dim=1)
    predClassTensor = torch.argmax(yPredTensor, dim=1)
    accuracyValue = (trueClassTensor == predClassTensor).float().mean().item()
    return accuracyValue

def getParameterMetadata(modelInstance):
    parameterNames = []
    parameterShapes = []
    parameterSizes = []

    for parameterName, parameterValue in modelInstance.named_parameters():
        parameterNames.append(parameterName)
        parameterShapes.append(parameterValue.shape)
        parameterSizes.append(parameterValue.numel())

    return parameterNames, parameterShapes, parameterSizes

def vectorToParameterDict(parameterVector, parameterNames, parameterShapes, parameterSizes):
    parameterDict = OrderedDict()
    startIndex = 0

    for parameterName, parameterShape, parameterSize in zip(parameterNames, parameterShapes, parameterSizes):
        endIndex = startIndex + parameterSize
        parameterDict[parameterName] = parameterVector[startIndex:endIndex].view(parameterShape)
        startIndex = endIndex

    return parameterDict

def predictWithVector(modelInstance, parameterVector, inputTensor, parameterNames, parameterShapes, parameterSizes):
    parameterDict = vectorToParameterDict(
        parameterVector,
        parameterNames,
        parameterShapes,
        parameterSizes
    )
    predictionTensor = functional_call(modelInstance, parameterDict, (inputTensor,))
    return predictionTensor

# Full Batch Gradient Descent Training Loop

15) Training Full Batch Gradient Descent

In [16]:
gradientDescentModel = buildSequentialModel()
_ = gradientDescentModel(xTrainTensor[:1])
gradientDescentModel.load_state_dict(initialStateDict)

epochCountGradientDescent = 150
learningRateValue = 0.5

gradientDescentHistory = {
    "epoch": [],
    "loss": [],
    "trainAccuracy": [],
    "testAccuracy": []
}

logger.info(f"\n{'='*80}")
logger.info("Training Full Batch Gradient Descent")
logger.info(f"Epochs: {epochCountGradientDescent}, Learning Rate: {learningRateValue}")
logger.info(f"{'='*80}")

gradientDescentStartTime = time.perf_counter()

for epochIndex in range(1, epochCountGradientDescent + 1):
    gradientDescentModel.zero_grad()

    yTrainPredTensor = gradientDescentModel(xTrainTensor)
    lossValue = categoricalCrossEntropy(yTrainTensor, yTrainPredTensor)
    lossValue.backward()

    with torch.no_grad():
        for parameterValue in gradientDescentModel.parameters():
            parameterValue -= learningRateValue * parameterValue.grad

        updatedTrainPredTensor = gradientDescentModel(xTrainTensor)
        updatedTestPredTensor = gradientDescentModel(xTestTensor)

        gradientDescentHistory["epoch"].append(epochIndex)
        gradientDescentHistory["loss"].append(categoricalCrossEntropy(yTrainTensor, updatedTrainPredTensor).item())
        gradientDescentHistory["trainAccuracy"].append(calculateAccuracy(yTrainTensor, updatedTrainPredTensor))
        gradientDescentHistory["testAccuracy"].append(calculateAccuracy(yTestTensor, updatedTestPredTensor))
        
        if epochIndex % 30 == 0:
            logger.info(f"Epoch {epochIndex}: Loss={gradientDescentHistory['loss'][-1]:.6f}, Train Acc={gradientDescentHistory['trainAccuracy'][-1]:.4f}, Test Acc={gradientDescentHistory['testAccuracy'][-1]:.4f}")

gradientDescentDuration = time.perf_counter() - gradientDescentStartTime

logger.info(f"Full Batch Gradient Descent training completed")
logger.info(f"Duration: {gradientDescentDuration:.4f} seconds")

2026-04-02 17:56:29,012 - 
2026-04-02 17:56:29,013 - Training Full Batch Gradient Descent
2026-04-02 17:56:29,013 - Epochs: 150, Learning Rate: 0.5
2026-04-02 17:56:29,014 - ================================================================================
2026-04-02 17:56:29,053 - Epoch 30: Loss=0.302854, Train Acc=0.9417, Test Acc=0.9333
2026-04-02 17:56:29,085 - Epoch 60: Loss=0.108617, Train Acc=0.9750, Test Acc=0.9667
2026-04-02 17:56:29,118 - Epoch 90: Loss=0.066551, Train Acc=0.9750, Test Acc=0.9667
2026-04-02 17:56:29,152 - Epoch 120: Loss=0.048557, Train Acc=0.9833, Test Acc=0.9667
2026-04-02 17:56:29,184 - Epoch 150: Loss=0.057400, Train Acc=0.9750, Test Acc=0.9667
2026-04-02 17:56:29,185 - Full Batch Gradient Descent training completed
2026-04-02 17:56:29,186 - Duration: 0.1703 seconds


# Gauss-Newton Method

16) Compute the Jacobian matrix

In [17]:
gaussNewtonModel = buildSequentialModel()
_ = gaussNewtonModel(xTrainTensor[:1])
gaussNewtonModel.load_state_dict(initialStateDict)

parameterNames, parameterShapes, parameterSizes = getParameterMetadata(gaussNewtonModel)
parameterVector = parameters_to_vector(gaussNewtonModel.parameters()).detach().clone().requires_grad_(True)

def residualFunction(currentParameterVector):
    yPredTensor = predictWithVector(
        gaussNewtonModel,
        currentParameterVector,
        xTrainTensor,
        parameterNames,
        parameterShapes,
        parameterSizes
    )
    residualTensor = (yTrainTensor - yPredTensor).reshape(-1)
    return residualTensor

residualVector = residualFunction(parameterVector)
jacobianMatrix = jacrev(residualFunction)(parameterVector)

print("Residual vector shape:", residualVector.shape)
print("Jacobian matrix shape:", jacobianMatrix.shape)

Residual vector shape: torch.Size([360])
Jacobian matrix shape: torch.Size([360, 115])


17) Compute the update step

In [18]:
dampingValue = 0.1
parameterCount = parameterVector.numel()

identityMatrix = torch.eye(parameterCount, dtype=parameterVector.dtype)
jtjMatrix = jacobianMatrix.T @ jacobianMatrix + dampingValue * identityMatrix
jtrVector = jacobianMatrix.T @ residualVector

try:
    deltaVector = -torch.linalg.solve(jtjMatrix, jtrVector)
except RuntimeError:
    deltaVector = -torch.linalg.pinv(jtjMatrix) @ jtrVector

print("Delta vector shape:", deltaVector.shape)
print("Norm delta:", torch.norm(deltaVector).item())

Delta vector shape: torch.Size([115])
Norm delta: 9.863125801086426


18) Apply parameter updates

In [19]:
updatedParameterVector = (parameterVector + deltaVector).detach()
vector_to_parameters(updatedParameterVector, gaussNewtonModel.parameters())

with torch.no_grad():
    yTrainPredAfterGnTensor = gaussNewtonModel(xTrainTensor)
    lossAfterGnValue = categoricalCrossEntropy(yTrainTensor, yTrainPredAfterGnTensor)

print("Loss after 1 Gauss-Newton step:", lossAfterGnValue.item())

Loss after 1 Gauss-Newton step: 3.1467387676239014


# Gauss-Newton Training Loop

19) Training Gauss-Newton

In [20]:
gaussNewtonModel = buildSequentialModel()
_ = gaussNewtonModel(xTrainTensor[:1])
gaussNewtonModel.load_state_dict(initialStateDict)

epochCountGaussNewton = 8
dampingValue = 0.1

gaussNewtonHistory = {
    "epoch": [],
    "loss": [],
    "trainAccuracy": [],
    "testAccuracy": []
}

parameterNames, parameterShapes, parameterSizes = getParameterMetadata(gaussNewtonModel)
parameterVector = parameters_to_vector(gaussNewtonModel.parameters()).detach().clone()

logger.info(f"\n{'='*80}")
logger.info("Training Gauss-Newton Method")
logger.info(f"Epochs: {epochCountGaussNewton}, Damping: {dampingValue}")
logger.info(f"{'='*80}")

gaussNewtonStartTime = time.perf_counter()

for epochIndex in range(1, epochCountGaussNewton + 1):
    parameterVector = parameterVector.detach().clone().requires_grad_(True)

    def residualFunction(currentParameterVector):
        yPredTensor = predictWithVector(
            gaussNewtonModel,
            currentParameterVector,
            xTrainTensor,
            parameterNames,
            parameterShapes,
            parameterSizes
        )
        residualTensor = (yTrainTensor - yPredTensor).reshape(-1)
        return residualTensor

    residualVector = residualFunction(parameterVector)
    jacobianMatrix = jacrev(residualFunction)(parameterVector)

    identityMatrix = torch.eye(parameterVector.numel(), dtype=parameterVector.dtype)
    jtjMatrix = jacobianMatrix.T @ jacobianMatrix + dampingValue * identityMatrix
    jtrVector = jacobianMatrix.T @ residualVector

    try:
        deltaVector = -torch.linalg.solve(jtjMatrix, jtrVector)
    except RuntimeError:
        deltaVector = -torch.linalg.pinv(jtjMatrix) @ jtrVector

    updatedParameterVector = (parameterVector + deltaVector).detach()
    vector_to_parameters(updatedParameterVector, gaussNewtonModel.parameters())
    parameterVector = updatedParameterVector

    with torch.no_grad():
        updatedTrainPredTensor = gaussNewtonModel(xTrainTensor)
        updatedTestPredTensor = gaussNewtonModel(xTestTensor)

        gaussNewtonHistory["epoch"].append(epochIndex)
        gaussNewtonHistory["loss"].append(categoricalCrossEntropy(yTrainTensor, updatedTrainPredTensor).item())
        gaussNewtonHistory["trainAccuracy"].append(calculateAccuracy(yTrainTensor, updatedTrainPredTensor))
        gaussNewtonHistory["testAccuracy"].append(calculateAccuracy(yTestTensor, updatedTestPredTensor))
        
        logger.info(f"Epoch {epochIndex}: Loss={gaussNewtonHistory['loss'][-1]:.6f}, Train Acc={gaussNewtonHistory['trainAccuracy'][-1]:.4f}, Test Acc={gaussNewtonHistory['testAccuracy'][-1]:.4f}")

gaussNewtonDuration = time.perf_counter() - gaussNewtonStartTime

logger.info(f"Gauss-Newton training completed")
logger.info(f"Duration: {gaussNewtonDuration:.4f} seconds")

2026-04-02 17:56:34,264 - 
2026-04-02 17:56:34,264 - Training Gauss-Newton Method
2026-04-02 17:56:34,265 - Epochs: 8, Damping: 0.1
2026-04-02 17:56:34,266 - ================================================================================
2026-04-02 17:56:34,275 - Epoch 1: Loss=3.146739, Train Acc=0.4500, Test Acc=0.4333
2026-04-02 17:56:34,282 - Epoch 2: Loss=0.833613, Train Acc=0.5917, Test Acc=0.6333
2026-04-02 17:56:34,291 - Epoch 3: Loss=0.677588, Train Acc=0.7250, Test Acc=0.7333
2026-04-02 17:56:34,297 - Epoch 4: Loss=5.118683, Train Acc=0.6667, Test Acc=0.6667
2026-04-02 17:56:34,303 - Epoch 5: Loss=5.364119, Train Acc=0.6667, Test Acc=0.6667
2026-04-02 17:56:34,309 - Epoch 6: Loss=5.376673, Train Acc=0.6667, Test Acc=0.6667
2026-04-02 17:56:34,316 - Epoch 7: Loss=5.380458, Train Acc=0.6667, Test Acc=0.6667
2026-04-02 17:56:34,322 - Epoch 8: Loss=5.378313, Train Acc=0.6667, Test Acc=0.6667
2026-04-02 17:56:34,322 - Gauss-Newton training completed
2026-04-02 17:56:34,323 - Durat

# Evaluation

20) Track loss per epoch

In [21]:
lossTrackingDataFrame = pd.DataFrame({
    "gradientDescentEpoch": gradientDescentHistory["epoch"],
    "gradientDescentLoss": gradientDescentHistory["loss"]
})

gaussNewtonLossDataFrame = pd.DataFrame({
    "gaussNewtonEpoch": gaussNewtonHistory["epoch"],
    "gaussNewtonLoss": gaussNewtonHistory["loss"]
})

display(lossTrackingDataFrame.head())
display(gaussNewtonLossDataFrame.head())

,gradientDescentEpoch,gradientDescentLoss
0,1,1.101141
1,2,1.086589
2,3,1.072145
3,4,1.055877
4,5,1.036264


,gaussNewtonEpoch,gaussNewtonLoss
0,1,3.146739
1,2,0.833613
2,3,0.677588
3,4,5.118683
4,5,5.364119


21) Compute training accuracy

In [22]:
with torch.no_grad():
    gradientDescentTrainPredTensor = gradientDescentModel(xTrainTensor)
    gaussNewtonTrainPredTensor = gaussNewtonModel(xTrainTensor)

gradientDescentTrainAccuracy = calculateAccuracy(yTrainTensor, gradientDescentTrainPredTensor)
gaussNewtonTrainAccuracy = calculateAccuracy(yTrainTensor, gaussNewtonTrainPredTensor)

print("Gradient Descent Train Accuracy:", gradientDescentTrainAccuracy)
print("Gauss-Newton Train Accuracy:", gaussNewtonTrainAccuracy)

Gradient Descent Train Accuracy: 0.9750000238418579
Gauss-Newton Train Accuracy: 0.6666666865348816


22) Compute testing accuracy

In [23]:
with torch.no_grad():
    gradientDescentTestPredTensor = gradientDescentModel(xTestTensor)
    gaussNewtonTestPredTensor = gaussNewtonModel(xTestTensor)

gradientDescentTestAccuracy = calculateAccuracy(yTestTensor, gradientDescentTestPredTensor)
gaussNewtonTestAccuracy = calculateAccuracy(yTestTensor, gaussNewtonTestPredTensor)

print("Gradient Descent Test Accuracy:", gradientDescentTestAccuracy)
print("Gauss-Newton Test Accuracy:", gaussNewtonTestAccuracy)

Gradient Descent Test Accuracy: 0.9666666388511658
Gauss-Newton Test Accuracy: 0.6666666865348816


# Visualization

23) Plot loss vs epoch

In [24]:
plt.figure(figsize=(10, 6))
plt.plot(gradientDescentHistory["epoch"], gradientDescentHistory["loss"], label="Full Batch Gradient Descent")
plt.plot(gaussNewtonHistory["epoch"], gaussNewtonHistory["loss"], label="Gauss-Newton")
plt.title("Loss vs Epoch")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plot_file = f"{output_dir}01_loss_vs_epoch.png"
plt.savefig(plot_file, dpi=300, bbox_inches='tight')
logger.info(f"Saved: {plot_file}")
plt.close()

2026-04-02 17:56:34,999 - Saved: visualizations/01_loss_vs_epoch.png


24) Plot accuracy vs epoch

In [25]:
plt.figure(figsize=(10, 6))
plt.plot(gradientDescentHistory["epoch"], gradientDescentHistory["trainAccuracy"], label="GD Train Accuracy")
plt.plot(gradientDescentHistory["epoch"], gradientDescentHistory["testAccuracy"], label="GD Test Accuracy")
plt.plot(gaussNewtonHistory["epoch"], gaussNewtonHistory["trainAccuracy"], label="GN Train Accuracy")
plt.plot(gaussNewtonHistory["epoch"], gaussNewtonHistory["testAccuracy"], label="GN Test Accuracy")
plt.title("Accuracy vs Epoch")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plot_file = f"{output_dir}02_accuracy_vs_epoch.png"
plt.savefig(plot_file, dpi=300, bbox_inches='tight')
logger.info(f"Saved: {plot_file}")
plt.close()

2026-04-02 17:56:35,486 - Saved: visualizations/02_accuracy_vs_epoch.png


25) Compare convergence speed

In [26]:
comparisonDataFrame = pd.DataFrame({
    "methodName": ["Full Batch Gradient Descent", "Gauss-Newton"],
    "epochCount": [epochCountGradientDescent, epochCountGaussNewton],
    "finalLoss": [
        gradientDescentHistory["loss"][-1],
        gaussNewtonHistory["loss"][-1]
    ],
    "finalTrainAccuracy": [
        gradientDescentHistory["trainAccuracy"][-1],
        gaussNewtonHistory["trainAccuracy"][-1]
    ],
    "finalTestAccuracy": [
        gradientDescentHistory["testAccuracy"][-1],
        gaussNewtonHistory["testAccuracy"][-1]
    ],
    "trainingTimeSecond": [
        gradientDescentDuration,
        gaussNewtonDuration
    ]
})

display(comparisonDataFrame)

,methodName,epochCount,finalLoss,finalTrainAccuracy,finalTestAccuracy,trainingTimeSecond
0,Full Batch Gradient Descent,150,0.057400,0.975000,0.966667,0.170278
1,Gauss-Newton,8,5.378313,0.666667,0.666667,0.055778


# Analysis Questions

26) Which method converges faster?

In [27]:
def getEpochToTargetAccuracy(accuracyList, targetAccuracy):
    for epochValue, accuracyValue in enumerate(accuracyList, start=1):
        if accuracyValue >= targetAccuracy:
            return epochValue
    return None

targetAccuracyValue = 0.90

gradientDescentEpochToTarget = getEpochToTargetAccuracy(
    gradientDescentHistory["testAccuracy"],
    targetAccuracyValue
)

gaussNewtonEpochToTarget = getEpochToTargetAccuracy(
    gaussNewtonHistory["testAccuracy"],
    targetAccuracyValue
)

logger.info(f"\n{'='*80}")
logger.info("ANALYSIS: Which method converges faster?")
logger.info(f"{'='*80}")
logger.info(f"GD epochs to reach target {targetAccuracyValue}: {gradientDescentEpochToTarget}")
logger.info(f"GN epochs to reach target {targetAccuracyValue}: {gaussNewtonEpochToTarget}")

if gradientDescentEpochToTarget is not None and gaussNewtonEpochToTarget is not None:
    if gaussNewtonEpochToTarget < gradientDescentEpochToTarget:
        logger.info("✓ Gauss-Newton converges faster.")
    elif gradientDescentEpochToTarget < gaussNewtonEpochToTarget:
        logger.info("✓ Full Batch Gradient Descent converges faster.")
    else:
        logger.info("✓ Both methods have the same convergence speed.")
else:
    logger.info("⚠ One of the methods did not reach target accuracy.")

2026-04-02 17:56:35,608 - 
2026-04-02 17:56:35,609 - ANALYSIS: Which method converges faster?
2026-04-02 17:56:35,609 - ================================================================================
2026-04-02 17:56:35,610 - GD epochs to reach target 0.9: 29
2026-04-02 17:56:35,611 - GN epochs to reach target 0.9: None
2026-04-02 17:56:35,612 - ⚠ One of the methods did not reach target accuracy.


27) Is Gauss-Newton stable?

In [28]:
def checkGaussNewtonStability(lossList):
    lossArray = np.array(lossList, dtype=np.float64)
    isFiniteValue = np.isfinite(lossArray).all()

    if len(lossArray) <= 1:
        return isFiniteValue

    maxJumpValue = np.max(np.abs(np.diff(lossArray)))
    return isFiniteValue and maxJumpValue < 15

gaussNewtonStable = checkGaussNewtonStability(gaussNewtonHistory["loss"])

logger.info(f"\n{'='*80}")
logger.info("ANALYSIS: Is Gauss-Newton stable?")
logger.info(f"{'='*80}")
logger.info(f"Gauss-Newton stable: {gaussNewtonStable}")

2026-04-02 17:56:35,662 - 
2026-04-02 17:56:35,662 - ANALYSIS: Is Gauss-Newton stable?
2026-04-02 17:56:35,663 - ================================================================================
2026-04-02 17:56:35,664 - Gauss-Newton stable: True


28) Compare computational complexity

In [29]:
parameterCountValue = sum(parameter.numel() for parameter in gradientDescentModel.parameters())
sampleCountValue = xTrainTensor.shape[0]
classCountValue = yTrainTensor.shape[1]
residualCountValue = sampleCountValue * classCountValue

logger.info(f"\n{'='*80}")
logger.info("ANALYSIS: Compare computational complexity")
logger.info(f"{'='*80}")
logger.info(f"Number of parameters: {parameterCountValue}")
logger.info(f"Number of residuals: {residualCountValue}")
logger.info(f"\nComplexity comparison:")
logger.info(f"Full Batch Gradient Descent: O(n x p) per epoch")
logger.info(f"Gauss-Newton: O(n x p^2 + p^3) per epoch")

2026-04-02 17:56:35,722 - 
2026-04-02 17:56:35,723 - ANALYSIS: Compare computational complexity
2026-04-02 17:56:35,724 - ================================================================================
2026-04-02 17:56:35,725 - Number of parameters: 115
2026-04-02 17:56:35,725 - Number of residuals: 360
2026-04-02 17:56:35,727 - 
Complexity comparison:
2026-04-02 17:56:35,728 - Full Batch Gradient Descent: O(n x p) per epoch
2026-04-02 17:56:35,729 - Gauss-Newton: O(n x p^2 + p^3) per epoch


29) When is Gauss-Newton preferable?

In [30]:
logger.info(f"\n{'='*80}")
logger.info("ANALYSIS: When is Gauss-Newton preferable?")
logger.info(f"{'='*80}")
logger.info("Gauss-Newton is more suitable when:")
logger.info("1. Dataset is relatively small or medium-sized")
logger.info("2. Number of model parameters is not too large")
logger.info("3. Want fast convergence in the number of epochs")
logger.info("4. Jacobian computation cost is still tolerable")
logger.info("5. Optimization problem approximates least-squares")

2026-04-02 17:56:35,785 - 
2026-04-02 17:56:35,786 - ANALYSIS: When is Gauss-Newton preferable?
2026-04-02 17:56:35,787 - ================================================================================
2026-04-02 17:56:35,788 - Gauss-Newton is more suitable when:
2026-04-02 17:56:35,788 - 1. Dataset is relatively small or medium-sized
2026-04-02 17:56:35,789 - 2. Number of model parameters is not too large
2026-04-02 17:56:35,790 - 3. Want fast convergence in the number of epochs
2026-04-02 17:56:35,791 - 4. Jacobian computation cost is still tolerable
2026-04-02 17:56:35,792 - 5. Optimization problem approximates least-squares


# Bonus Tasks

30) Add regularization

In [31]:
l2LambdaValue = 0.001
epochCountRegularized = 150
learningRateRegularized = 0.5

def categoricalCrossEntropyWithRegularization(yTrueTensor, yPredTensor, modelInstance, l2LambdaValue):
    baseLossValue = categoricalCrossEntropy(yTrueTensor, yPredTensor)

    l2PenaltyValue = torch.tensor(0.0, dtype=baseLossValue.dtype)
    for parameterName, parameterValue in modelInstance.named_parameters():
        if "weight" in parameterName:
            l2PenaltyValue = l2PenaltyValue + torch.sum(parameterValue ** 2)

    totalLossValue = baseLossValue + l2LambdaValue * l2PenaltyValue
    return totalLossValue

regularizedModel = buildSequentialModel()
_ = regularizedModel(xTrainTensor[:1])
regularizedModel.load_state_dict(initialStateDict)

regularizedHistory = {
    "epoch": [],
    "loss": [],
    "trainAccuracy": [],
    "testAccuracy": []
}

for epochIndex in range(1, epochCountRegularized + 1):
    regularizedModel.zero_grad()

    yTrainPredTensor = regularizedModel(xTrainTensor)
    totalLossValue = categoricalCrossEntropyWithRegularization(
        yTrainTensor,
        yTrainPredTensor,
        regularizedModel,
        l2LambdaValue
    )
    totalLossValue.backward()

    with torch.no_grad():
        for parameterValue in regularizedModel.parameters():
            parameterValue -= learningRateRegularized * parameterValue.grad

        updatedTrainPredTensor = regularizedModel(xTrainTensor)
        updatedTestPredTensor = regularizedModel(xTestTensor)

        regularizedHistory["epoch"].append(epochIndex)
        regularizedHistory["loss"].append(
            categoricalCrossEntropyWithRegularization(
                yTrainTensor,
                updatedTrainPredTensor,
                regularizedModel,
                l2LambdaValue
            ).item()
        )
        regularizedHistory["trainAccuracy"].append(calculateAccuracy(yTrainTensor, updatedTrainPredTensor))
        regularizedHistory["testAccuracy"].append(calculateAccuracy(yTestTensor, updatedTestPredTensor))

print("Regularization completed")
print("Final Regularized Train Accuracy:", regularizedHistory["trainAccuracy"][-1])
print("Final Regularized Test Accuracy:", regularizedHistory["testAccuracy"][-1])
print("Final Regularized Loss:", regularizedHistory["loss"][-1])

Regularization completed
Final Regularized Train Accuracy: 0.9833333492279053
Final Regularized Test Accuracy: 1.0
Final Regularized Loss: 0.08778227865695953


31) Compare with built-in optimizers

In [32]:
def trainWithBuiltInOptimizer(optimizerName, learningRateValue, epochCount):
    modelInstance = buildSequentialModel()
    _ = modelInstance(xTrainTensor[:1])
    modelInstance.load_state_dict(initialStateDict)

    if optimizerName == "SGD":
        optimizerInstance = torch.optim.SGD(
            modelInstance.parameters(),
            lr=learningRateValue,
            momentum=0.9
        )
    elif optimizerName == "Adam":
        optimizerInstance = torch.optim.Adam(
            modelInstance.parameters(),
            lr=learningRateValue
        )
    elif optimizerName == "AdamW":
        optimizerInstance = torch.optim.AdamW(
            modelInstance.parameters(),
            lr=learningRateValue,
            weight_decay=1e-4
        )
    elif optimizerName == "RMSprop":
        optimizerInstance = torch.optim.RMSprop(
            modelInstance.parameters(),
            lr=learningRateValue
        )
    elif optimizerName == "Adagrad":
        optimizerInstance = torch.optim.Adagrad(
            modelInstance.parameters(),
            lr=learningRateValue
        )
    elif optimizerName == "NAdam":
        optimizerInstance = torch.optim.NAdam(
            modelInstance.parameters(),
            lr=learningRateValue
        )
    else:
        raise ValueError("Optimizer not supported")

    historyDict = {
        "epoch": [],
        "loss": [],
        "trainAccuracy": [],
        "testAccuracy": []
    }

    for epochIndex in range(1, epochCount + 1):
        optimizerInstance.zero_grad()

        yTrainPredTensor = modelInstance(xTrainTensor)
        lossValue = categoricalCrossEntropy(yTrainTensor, yTrainPredTensor)
        lossValue.backward()
        optimizerInstance.step()

        with torch.no_grad():
            updatedTrainPredTensor = modelInstance(xTrainTensor)
            updatedTestPredTensor = modelInstance(xTestTensor)

            historyDict["epoch"].append(epochIndex)
            historyDict["loss"].append(categoricalCrossEntropy(yTrainTensor, updatedTrainPredTensor).item())
            historyDict["trainAccuracy"].append(calculateAccuracy(yTrainTensor, updatedTrainPredTensor))
            historyDict["testAccuracy"].append(calculateAccuracy(yTestTensor, updatedTestPredTensor))

    return modelInstance, historyDict

optimizerConfigDict = {
    "SGD": 0.1,
    "Adam": 0.01,
    "AdamW": 0.01,
    "RMSprop": 0.01,
    "Adagrad": 0.05,
    "NAdam": 0.01
}

optimizerHistoryDict = {}

logger.info(f"\n{'='*80}")
logger.info("Training with Built-in Optimizers")
logger.info(f"{'='*80}")

for optimizerName, learningRateValue in optimizerConfigDict.items():
    logger.info(f"Training {optimizerName} with learning rate={learningRateValue}...")
    _, historyDict = trainWithBuiltInOptimizer(
        optimizerName=optimizerName,
        learningRateValue=learningRateValue,
        epochCount=150
    )
    optimizerHistoryDict[optimizerName] = historyDict
    logger.info(f"  Final Loss: {historyDict['loss'][-1]:.6f}, Final Test Acc: {historyDict['testAccuracy'][-1]:.4f}")

builtInOptimizerComparisonDataFrame = pd.DataFrame({
    "methodName": [
        "Manual Full Batch Gradient Descent",
        "Built-in SGD",
        "Built-in Adam",
        "Built-in AdamW",
        "Built-in RMSprop",
        "Built-in Adagrad",
        "Built-in NAdam"
    ],
    "finalLoss": [
        gradientDescentHistory["loss"][-1],
        optimizerHistoryDict["SGD"]["loss"][-1],
        optimizerHistoryDict["Adam"]["loss"][-1],
        optimizerHistoryDict["AdamW"]["loss"][-1],
        optimizerHistoryDict["RMSprop"]["loss"][-1],
        optimizerHistoryDict["Adagrad"]["loss"][-1],
        optimizerHistoryDict["NAdam"]["loss"][-1]
    ],
    "finalTrainAccuracy": [
        gradientDescentHistory["trainAccuracy"][-1],
        optimizerHistoryDict["SGD"]["trainAccuracy"][-1],
        optimizerHistoryDict["Adam"]["trainAccuracy"][-1],
        optimizerHistoryDict["AdamW"]["trainAccuracy"][-1],
        optimizerHistoryDict["RMSprop"]["trainAccuracy"][-1],
        optimizerHistoryDict["Adagrad"]["trainAccuracy"][-1],
        optimizerHistoryDict["NAdam"]["trainAccuracy"][-1]
    ],
    "finalTestAccuracy": [
        gradientDescentHistory["testAccuracy"][-1],
        optimizerHistoryDict["SGD"]["testAccuracy"][-1],
        optimizerHistoryDict["Adam"]["testAccuracy"][-1],
        optimizerHistoryDict["AdamW"]["testAccuracy"][-1],
        optimizerHistoryDict["RMSprop"]["testAccuracy"][-1],
        optimizerHistoryDict["Adagrad"]["testAccuracy"][-1],
        optimizerHistoryDict["NAdam"]["testAccuracy"][-1]
    ]
})

display(builtInOptimizerComparisonDataFrame)

plt.figure(figsize=(12, 6))
plt.plot(gradientDescentHistory["epoch"], gradientDescentHistory["loss"], label="Manual GD", linewidth=2)
plt.plot(optimizerHistoryDict["SGD"]["epoch"], optimizerHistoryDict["SGD"]["loss"], label="SGD")
plt.plot(optimizerHistoryDict["Adam"]["epoch"], optimizerHistoryDict["Adam"]["loss"], label="Adam")
plt.plot(optimizerHistoryDict["AdamW"]["epoch"], optimizerHistoryDict["AdamW"]["loss"], label="AdamW")
plt.plot(optimizerHistoryDict["RMSprop"]["epoch"], optimizerHistoryDict["RMSprop"]["loss"], label="RMSprop")
plt.plot(optimizerHistoryDict["Adagrad"]["epoch"], optimizerHistoryDict["Adagrad"]["loss"], label="Adagrad")
plt.plot(optimizerHistoryDict["NAdam"]["epoch"], optimizerHistoryDict["NAdam"]["loss"], label="NAdam")

plt.title("Loss Comparison with Built-in Optimizers")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plot_file = f"{output_dir}03_builtin_optimizers_comparison.png"
plt.savefig(plot_file, dpi=300, bbox_inches='tight')
logger.info(f"Saved: {plot_file}")
plt.close()

2026-04-02 17:56:36,166 - 
2026-04-02 17:56:36,167 - Training with Built-in Optimizers
2026-04-02 17:56:36,168 - ================================================================================
2026-04-02 17:56:36,169 - Training SGD with learning rate=0.1...
2026-04-02 17:56:36,351 -   Final Loss: 0.037491, Final Test Acc: 0.9667
2026-04-02 17:56:36,351 - Training Adam with learning rate=0.01...
2026-04-02 17:56:36,590 -   Final Loss: 0.090553, Final Test Acc: 0.9667
2026-04-02 17:56:36,590 - Training AdamW with learning rate=0.01...
2026-04-02 17:56:36,827 -   Final Loss: 0.090562, Final Test Acc: 0.9667
2026-04-02 17:56:36,828 - Training RMSprop with learning rate=0.01...
2026-04-02 17:56:37,044 -   Final Loss: 0.053776, Final Test Acc: 0.9667
2026-04-02 17:56:37,045 - Training Adagrad with learning rate=0.05...
2026-04-02 17:56:37,251 -   Final Loss: 0.107018, Final Test Acc: 0.9667
2026-04-02 17:56:37,252 - Training NAdam with learning rate=0.01...
2026-04-02 17:56:37,499 -   Final

,methodName,finalLoss,finalTrainAccuracy,finalTestAccuracy
0,Manual Full Batch Gradient Descent,0.057400,0.975000,0.966667
1,Built-in SGD,0.037491,0.983333,0.966667
2,Built-in Adam,0.090553,0.983333,0.966667
3,Built-in AdamW,0.090562,0.983333,0.966667
4,Built-in RMSprop,0.053776,0.983333,0.966667
5,Built-in Adagrad,0.107018,0.983333,0.966667
6,Built-in NAdam,0.085014,0.983333,0.966667


2026-04-02 17:56:38,002 - Saved: visualizations/03_builtin_optimizers_comparison.png


32) Experiment with different learning rates

In [33]:
learningRateList = [0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 1.0]
epochCountLearningRateExperiment = 100

learningRateExperimentResultList = []

logger.info(f"\n{'='*80}")
logger.info("Learning Rate Experiment")
logger.info(f"{'='*80}")

for learningRateValue in learningRateList:
    modelInstance = buildSequentialModel()
    _ = modelInstance(xTrainTensor[:1])
    modelInstance.load_state_dict(initialStateDict)

    historyDict = {
        "epoch": [],
        "loss": [],
        "trainAccuracy": [],
        "testAccuracy": []
    }

    for epochIndex in range(1, epochCountLearningRateExperiment + 1):
        modelInstance.zero_grad()

        yTrainPredTensor = modelInstance(xTrainTensor)
        lossValue = categoricalCrossEntropy(yTrainTensor, yTrainPredTensor)
        lossValue.backward()

        with torch.no_grad():
            for parameterValue in modelInstance.parameters():
                parameterValue -= learningRateValue * parameterValue.grad

            updatedTrainPredTensor = modelInstance(xTrainTensor)
            updatedTestPredTensor = modelInstance(xTestTensor)

            historyDict["epoch"].append(epochIndex)
            historyDict["loss"].append(categoricalCrossEntropy(yTrainTensor, updatedTrainPredTensor).item())
            historyDict["trainAccuracy"].append(calculateAccuracy(yTrainTensor, updatedTrainPredTensor))
            historyDict["testAccuracy"].append(calculateAccuracy(yTestTensor, updatedTestPredTensor))

    learningRateExperimentResultList.append({
        "learningRate": learningRateValue,
        "finalLoss": historyDict["loss"][-1],
        "finalTrainAccuracy": historyDict["trainAccuracy"][-1],
        "finalTestAccuracy": historyDict["testAccuracy"][-1],
        "history": historyDict
    })
    
    logger.info(f"LR={learningRateValue}: Loss={historyDict['loss'][-1]:.6f}, Train Acc={historyDict['trainAccuracy'][-1]:.4f}, Test Acc={historyDict['testAccuracy'][-1]:.4f}")

learningRateResultDataFrame = pd.DataFrame([
    {
        "learningRate": resultItem["learningRate"],
        "finalLoss": resultItem["finalLoss"],
        "finalTrainAccuracy": resultItem["finalTrainAccuracy"],
        "finalTestAccuracy": resultItem["finalTestAccuracy"]
    }
    for resultItem in learningRateExperimentResultList
])

display(learningRateResultDataFrame)

plt.figure(figsize=(12, 6))
for resultItem in learningRateExperimentResultList:
    plt.plot(
        resultItem["history"]["epoch"],
        resultItem["history"]["loss"],
        label=f"lr={resultItem['learningRate']}"
    )

plt.title("Learning Rate Experiment - Loss vs Epoch")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plot_file = f"{output_dir}04_learning_rate_experiment.png"
plt.savefig(plot_file, dpi=300, bbox_inches='tight')
logger.info(f"Saved: {plot_file}")
plt.close()

2026-04-02 17:56:38,063 - 
2026-04-02 17:56:38,063 - Learning Rate Experiment
2026-04-02 17:56:38,064 - ================================================================================
2026-04-02 17:56:38,180 - LR=0.01: Loss=1.086360, Train Acc=0.3333, Test Acc=0.3333
2026-04-02 17:56:38,289 - LR=0.05: Loss=0.816342, Train Acc=0.7167, Test Acc=0.7667
2026-04-02 17:56:38,396 - LR=0.1: Loss=0.468435, Train Acc=0.8833, Test Acc=0.8333
2026-04-02 17:56:38,512 - LR=0.25: Loss=0.132999, Train Acc=0.9750, Test Acc=0.9667
2026-04-02 17:56:38,622 - LR=0.5: Loss=0.058181, Train Acc=0.9833, Test Acc=0.9667
2026-04-02 17:56:38,734 - LR=0.75: Loss=0.043013, Train Acc=0.9833, Test Acc=0.9667
2026-04-02 17:56:38,840 - LR=1.0: Loss=0.041073, Train Acc=0.9833, Test Acc=1.0000


,learningRate,finalLoss,finalTrainAccuracy,finalTestAccuracy
0,0.01,1.086360,0.333333,0.333333
1,0.05,0.816342,0.716667,0.766667
2,0.10,0.468435,0.883333,0.833333
3,0.25,0.132999,0.975000,0.966667
4,0.50,0.058181,0.983333,0.966667
5,0.75,0.043013,0.983333,0.966667
6,1.00,0.041073,0.983333,1.000000


2026-04-02 17:56:39,343 - Saved: visualizations/04_learning_rate_experiment.png


# Make Report

In [34]:
report_markdown = f"""# Iris Classification - Optimization in Neural Networks

## Dataset Information

| Attribute | Value |
|---|---|
| Dataset Name | Iris |
| Total Samples | {len(xNormalizedArray)} |
| Training Samples | {len(xTrainArray)} |
| Testing Samples | {len(xTestArray)} |
| Number of Features | {xNormalizedArray.shape[1]} |
| Number of Classes | {classCount} |
| Class Names | {', '.join(classNames)} |
| Test Split | 20% |

## Model Architecture

| Component | Specification |
|---|---|
| Input Layer | 4 neurons |
| Hidden Layer 1 | 8 neurons (ReLU) |
| Hidden Layer 2 | 6 neurons (ReLU) |
| Output Layer | 3 neurons (Softmax) |
| Total Parameters | {parameterCountValue} |
| Loss Function | Categorical Cross-Entropy |

## Training Results - Full Batch Gradient Descent

| Metric | Value |
|---|---|
| Epochs | {epochCountGradientDescent} |
| Learning Rate | {learningRateValue} |
| Initial Loss | {initialLossValue.item():.6f} |
| Final Loss | {gradientDescentHistory['loss'][-1]:.6f} |
| Final Train Accuracy | {gradientDescentHistory['trainAccuracy'][-1]:.4f} |
| Final Test Accuracy | {gradientDescentHistory['testAccuracy'][-1]:.4f} |
| Training Duration | {gradientDescentDuration:.4f} seconds |
| Convergence (90%) | {gradientDescentEpochToTarget} epochs |

## Training Results - Gauss-Newton Method

| Metric | Value |
|---|---|
| Epochs | {epochCountGaussNewton} |
| Damping Value | {dampingValue} |
| Final Loss | {gaussNewtonHistory['loss'][-1]:.6f} |
| Final Train Accuracy | {gaussNewtonHistory['trainAccuracy'][-1]:.4f} |
| Final Test Accuracy | {gaussNewtonHistory['testAccuracy'][-1]:.4f} |
| Training Duration | {gaussNewtonDuration:.4f} seconds |
| Convergence (90%) | {gaussNewtonEpochToTarget} epochs |
| Stability Status | {'Stable' if gaussNewtonStable else 'Unstable'} |

## Comparative Analysis

### Convergence Speed vs Training Time

- **Faster Convergence (epochs)**: {'Gauss-Newton' if gaussNewtonEpochToTarget and gradientDescentEpochToTarget and gaussNewtonEpochToTarget < gradientDescentEpochToTarget else 'Gradient Descent'}
- **Faster Training Time**: {'Gauss-Newton' if gaussNewtonDuration < gradientDescentDuration else 'Gradient Descent'}

### GD vs GN Convergence

| Metric | Gradient Descent | Gauss-Newton | Difference |
|---|---|---|---|
| Epochs to 90% accuracy | {gradientDescentEpochToTarget} | {gaussNewtonEpochToTarget} | {abs((gaussNewtonEpochToTarget or 0) - (gradientDescentEpochToTarget or 0))} |

### GD vs GN Training Time

| Metric | Value |
|---|---|
| GD Training Time | {gradientDescentDuration:.4f} seconds |
| GN Training Time | {gaussNewtonDuration:.4f} seconds |
| Time Difference | {abs(gradientDescentDuration - gaussNewtonDuration):.4f} seconds |

### Computational Complexity

| Method | Complexity |
|---|---|
| Gradient Descent | O(n × p) per epoch |
| Gauss-Newton | O(n × p² + p³) per epoch |
| Trade-off | GN faster per iteration but higher computational cost |

## Key Insights

### 1. Convergence Speed
{'Gauss-Newton converges significantly faster.' if (gaussNewtonEpochToTarget and gradientDescentEpochToTarget and gaussNewtonEpochToTarget < gradientDescentEpochToTarget) else 'Gradient Descent converges faster despite more iterations.'}

### 2. Stability
Gauss-Newton is **{'stable' if gaussNewtonStable else 'unstable'}** for this problem.

### 3. Efficiency Recommendation
For this Iris dataset, **{('Gauss-Newton' if (gaussNewtonDuration < gradientDescentDuration) else 'Gradient Descent')}** is recommended.

### 4. When to Use Gauss-Newton
- Dataset is relatively small to medium-sized
- Number of parameters is manageable
- Need fast convergence in epochs
- Jacobian computation cost is tolerable

## Regularization Impact

| Metric | Value |
|---|---|
| L2 Lambda | {l2LambdaValue} |
| Final Regularized Loss | {regularizedHistory['loss'][-1]:.6f} |
| Regularized Train Accuracy | {regularizedHistory['trainAccuracy'][-1]:.4f} |
| Regularized Test Accuracy | {regularizedHistory['testAccuracy'][-1]:.4f} |
| Impact | Reduces overfitting |

## Optimizer Benchmark Results

| Method | Loss | Train Accuracy | Test Accuracy |
|---|---|---|---|
| Manual Full Batch GD | {gradientDescentHistory['loss'][-1]:.6f} | {gradientDescentHistory['trainAccuracy'][-1]:.4f} | {gradientDescentHistory['testAccuracy'][-1]:.4f} |
| SGD | {optimizerHistoryDict['SGD']['loss'][-1]:.6f} | {optimizerHistoryDict['SGD']['trainAccuracy'][-1]:.4f} | {optimizerHistoryDict['SGD']['testAccuracy'][-1]:.4f} |
| Adam | {optimizerHistoryDict['Adam']['loss'][-1]:.6f} | {optimizerHistoryDict['Adam']['trainAccuracy'][-1]:.4f} | {optimizerHistoryDict['Adam']['testAccuracy'][-1]:.4f} |
| AdamW | {optimizerHistoryDict['AdamW']['loss'][-1]:.6f} | {optimizerHistoryDict['AdamW']['trainAccuracy'][-1]:.4f} | {optimizerHistoryDict['AdamW']['testAccuracy'][-1]:.4f} |
| RMSprop | {optimizerHistoryDict['RMSprop']['loss'][-1]:.6f} | {optimizerHistoryDict['RMSprop']['trainAccuracy'][-1]:.4f} | {optimizerHistoryDict['RMSprop']['testAccuracy'][-1]:.4f} |
| Adagrad | {optimizerHistoryDict['Adagrad']['loss'][-1]:.6f} | {optimizerHistoryDict['Adagrad']['trainAccuracy'][-1]:.4f} | {optimizerHistoryDict['Adagrad']['testAccuracy'][-1]:.4f} |
| NAdam | {optimizerHistoryDict['NAdam']['loss'][-1]:.6f} | {optimizerHistoryDict['NAdam']['trainAccuracy'][-1]:.4f} | {optimizerHistoryDict['NAdam']['testAccuracy'][-1]:.4f} |

## Learning Rate Analysis

| Metric | Value |
|---|---|
| Tested Learning Rates | {', '.join(str(lr) for lr in learningRateList)} |
| Optimal Learning Rate | {max(learningRateExperimentResultList, key=lambda x: x['finalTestAccuracy'])['learningRate']} |
| Best Test Accuracy | {max(learningRateExperimentResultList, key=lambda x: x['finalTestAccuracy'])['finalTestAccuracy']:.4f} |

## Generated Files

### Output Structure

**Output Directory**: {output_dir}

### Files List

- **logs.txt** - Training logs
- **config.json** - Model configuration
- **metrics.json** - Performance metrics
- **gd_training_history.csv** - Gradient Descent training data
- **gn_training_history.csv** - Gauss-Newton training data
- **optimizer_comparison.csv** - Optimizer benchmark results
- **learning_rate_experiment.csv** - Learning rate experiment results
- **method_comparison.csv** - Method comparison results

### Visualizations

1. **01_loss_vs_epoch.png** - Loss vs Epoch comparison
2. **02_accuracy_vs_epoch.png** - Accuracy vs Epoch comparison
3. **03_builtin_optimizers_comparison.png** - Optimizer comparison plot
4. **04_learning_rate_experiment.png** - Learning rate analysis plot

## Conclusion

This analysis demonstrates the trade-offs between **Gradient Descent** and **Gauss-Newton** optimization methods on the Iris classification task. While Gauss-Newton converges faster in terms of epochs, both methods achieve comparable final performance. The choice between them depends on the specific requirements of computational efficiency versus convergence speed.

For small to medium-sized datasets with manageable parameter counts, Gauss-Newton can be a compelling alternative to traditional gradient descent methods.

---

**Generated**: {datetime.now()}

"""

report_path = f"{output_dir}REPORT.md"
with open(report_path, 'w') as f:
    f.write(report_markdown)

logger.info(f"Saved markdown report: {report_path}")


2026-04-02 17:56:39,407 - Saved markdown report: visualizations/REPORT.md
